# 12장. 트랜스포머

셀프 어텐션을 넘파이로 계산하고, 위치 정보와 트랜스포머 블록을 더해 영화평을 분류한다. 같은 블록으로 다음 글자를 예측하는 작은 언어 모델을 만들고 문장을 생성한다.

**준비:** 11장과 같은 파일 `ratings_train.txt`(https://github.com/e9t/nsmc)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 시험용 파일 `ratings_test.txt`는 16장에서 쓴다. 끝까지 실행하는 데 20분쯤 걸린다.

## 12.1 셀프 어텐션과 여러 개의 헤드

In [ ]:
import math, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)
raw = pd.read_csv("ratings_train.txt", sep="\t").dropna(subset=["document"])
raw = raw.reset_index(drop=True)
order = np.random.default_rng(0).permutation(len(raw))        # 11장과 같은 나누기
train, valid = raw.iloc[order[:30000]], raw.iloc[order[-10000:]]
texts_tr, texts_va = train["document"].tolist(), valid["document"].tolist()
y_tr = torch.tensor(train["label"].to_numpy())
y_va = torch.tensor(valid["label"].to_numpy())

PAD, UNKNOWN = 0, 1                                 # 0번은 패딩, 1번은 어휘에 없는 글자
counts = Counter(ch for text in texts_tr for ch in text)
vocab = {ch: k + 2 for k, (ch, n) in enumerate(counts.most_common()) if n >= 5}
V = len(vocab) + 2

def encode(texts, length=64):
    """문장마다 글자 번호를 length칸에 적는다. 남는 칸은 0, 넘치는 글자는 버린다."""
    ids = torch.zeros(len(texts), length, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [vocab.get(ch, UNKNOWN) for ch in text][:length]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

X_tr, X_va = encode(texts_tr), encode(texts_va)
print(len(texts_tr), len(texts_va), V, X_tr.shape, X_va.shape)

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    return np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)

def attention(Q, K, V_, hide=None):
    """11장의 어텐션. hide가 True인 칸은 가린다."""
    scores = Q @ K.T / np.sqrt(K.shape[1])
    if hide is not None:
        scores = np.where(hide, -np.inf, scores)
    weights = softmax(scores)
    return weights @ V_, weights

X = np.array([[1.0, 0.0, 1.0, 0.0],                  # 토큰 3개, 토큰마다 숫자 4개
              [0.0, 2.0, 0.0, 0.0],
              [1.0, 1.0, 0.0, 0.0]])
out, weights = attention(X, X, X)                    # 질의, 키, 값이 모두 같은 X
print(X @ X.T)
print(weights.round(4))
print(out.round(4))

In [ ]:
def self_attention(X, Wq, Wk, Wv, Wo, heads, hide=None):
    """X: (토큰 수, d). 질의, 키, 값을 만들고 d를 heads개로 나눠 따로 어텐션한다."""
    size = X.shape[1] // heads                           # 헤드 하나가 쓰는 길이
    Q, K, V_ = X @ Wq.T, X @ Wk.T, X @ Wv.T              # 각각 (토큰 수, d)
    parts, maps = [], []
    for h in range(heads):
        cols = slice(h * size, (h + 1) * size)
        out, weights = attention(Q[:, cols], K[:, cols], V_[:, cols], hide)
        parts.append(out)                                # (토큰 수, size)
        maps.append(weights)                             # (토큰 수, 토큰 수)
    return np.concatenate(parts, axis=1) @ Wo.T, np.stack(maps)

I = np.eye(4)
out, maps = self_attention(X, I, I, I, I, heads=2)       # 가중치가 단위 행렬인 경우
print(maps.shape, out.shape)
print(maps[0].round(4))
print(maps[1].round(4))

rng = np.random.default_rng(0)
Wq, Wk, Wv, Wo = (rng.normal(0, 0.5, (4, 4)) for _ in range(4))
out, maps = self_attention(X, Wq, Wk, Wv, Wo, heads=2)
layer = nn.MultiheadAttention(4, 2, bias=False, batch_first=True, dtype=torch.float64)
with torch.no_grad():
    layer.in_proj_weight.copy_(torch.tensor(np.concatenate([Wq, Wk, Wv])))
    layer.out_proj.weight.copy_(torch.tensor(Wo))
    x = torch.tensor(X)[None]                            # (1, 3, 4)
    by_torch, by_torch_maps = layer(x, x, x, average_attn_weights=False)
print(float(np.abs(by_torch[0].numpy() - out).max()),
      float(np.abs(by_torch_maps[0].numpy() - maps).max()))

In [ ]:
class SelfAttention(nn.Module):
    """여러 헤드의 셀프 어텐션. x: (배치, 길이, dim)."""
    def __init__(self, dim, heads, scale=True):
        super().__init__()
        self.heads, self.scale = heads, scale
        self.to_q, self.to_k, self.to_v = (nn.Linear(dim, dim) for _ in range(3))
        self.out = nn.Linear(dim, dim)

    def forward(self, x, hide=None, causal=False):
        B, L, D = x.shape
        H = self.heads
        split = lambda t: t.view(B, L, H, D // H).transpose(1, 2)    # (B, H, L, D/H)
        q, k, v = split(self.to_q(x)), split(self.to_k(x)), split(self.to_v(x))
        scores = q @ k.transpose(-2, -1)                             # (B, H, L, L)
        if self.scale:
            scores = scores / math.sqrt(D // H)
        if hide is not None:                                        # 패딩인 키를 가린다
            scores = scores.masked_fill(hide[:, None, None, :], float("-inf"))
        if causal:                                                  # 뒤의 토큰을 가린다
            later = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(later, float("-inf"))
        self.maps = torch.softmax(scores, dim=-1)                   # 가중치를 남겨 둔다
        mixed = (self.maps @ v).transpose(1, 2).reshape(B, L, D)    # 헤드를 다시 붙인다
        return self.out(mixed)

torch.manual_seed(0)
layer = SelfAttention(64, 4)
x = torch.randn(8, 20, 64)                                         # 문장 8개, 토큰 20개
print(layer(x).shape, layer.maps.shape, sum(p.numel() for p in layer.parameters()))
print(layer.maps[0, 0].sum(dim=-1)[:4].detach().numpy().round(4))

## 12.2 위치 정보를 입력에 더하기

In [ ]:
swap = [2, 1, 0]                                         # 첫 토큰과 셋째 토큰을 바꾼다
out, maps = self_attention(X, Wq, Wk, Wv, Wo, heads=2)
out_swapped, _ = self_attention(X[swap], Wq, Wk, Wv, Wo, heads=2)
print(out.round(4))
print(out_swapped.round(4))
print(float(np.abs(out_swapped - out[swap]).max()),
      float(np.abs(out_swapped.mean(axis=0) - out.mean(axis=0)).max()))

In [ ]:
def make_task(n, seed):
    """숫자(0~9) 여섯 개를 여섯 칸에 놓는다. 질문은 칸 번호, 답은 그 칸의 숫자다."""
    rng = np.random.default_rng(seed)
    numbers = rng.integers(0, 10, (n, 6))
    where = rng.integers(0, 6, n)                        # 질문한 칸 (0~5)
    ids = np.column_stack([11 + where, 1 + numbers])     # 0번은 패딩이라 쓰지 않는다
    answer = numbers[np.arange(n), where]
    return torch.tensor(ids), torch.tensor(answer), torch.tensor(where), numbers

T_tr, a_tr, _, _ = make_task(20000, seed=0)
T_te, a_te, where_te, numbers_te = make_task(5000, seed=1)
print(T_tr.shape, T_tr[0].tolist(), a_tr[0].item())
print("질문: 항목", T_tr[0, 0].item() - 10, " 항목의 숫자:", (T_tr[0, 1:] - 1).tolist())

In [ ]:
def sinusoid(length, dim):
    """사인과 코사인으로 만든 위치 인코딩. (length, dim)"""
    pos = np.arange(length)[:, None]
    rate = 1 / 10000 ** (np.arange(0, dim, 2) / dim)     # 짝수 번째 칸마다 다른 빠르기
    table = np.zeros((length, dim))
    table[:, 0::2], table[:, 1::2] = np.sin(pos * rate), np.cos(pos * rate)
    return table

class TaskNet(nn.Module):
    """임베딩 → (위치 더하기) → 셀프 어텐션 한 번 → 평균 → 선형층."""
    def __init__(self, position, dim=32):
        super().__init__()
        self.table = nn.Embedding(17, dim)
        self.position = position
        if position == "learned":
            self.where = nn.Parameter(torch.randn(7, dim))               # 학습되는 표
        if position == "sinusoid":
            fixed = torch.tensor(sinusoid(7, dim), dtype=torch.float32)
            self.register_buffer("where", fixed)                          # 고정된 표
        self.attend = SelfAttention(dim, heads=1)
        self.head = nn.Linear(dim, 10)

    def forward(self, ids):
        x = self.table(ids)                                  # (배치, 7, 32)
        if self.position != "none":
            x = x + self.where                               # 칸마다 다른 벡터를 더한다
        return self.head(self.attend(x).mean(dim=1))

def accuracy(net, ids, y):
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(ids[k:k + 1000]).argmax(dim=1)
                           for k in range(0, len(ids), 1000)])
    return (guess == y).float().mean().item() * 100

def batches(lengths, seed, size=64, chunk=50):
    """길이가 비슷한 문장끼리 배치를 만든다. 섞은 뒤 3,200문장씩 길이 순으로 놓는다."""
    shuffler = torch.Generator().manual_seed(seed)
    mixed = torch.randperm(len(lengths), generator=shuffler)
    made = []
    for start in range(0, len(mixed), size * chunk):
        part = mixed[start:start + size * chunk]
        part = part[torch.argsort(lengths[part])]
        made += [part[k:k + size] for k in range(0, len(part), size)]
    return [made[k] for k in torch.randperm(len(made), generator=shuffler)]

def fit(net, X, y, lr, seed=0, epochs=2, loss_of=None, warm=200, watch=None):
    """AdamW, 배치 크기 64. 학습률은 warm걸음 동안 올리고 그 뒤 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    lengths = (X != PAD).sum(dim=1)
    total, step = epochs * math.ceil(len(X) / 64), 0
    for epoch in range(epochs):
        net.train()
        for rows in batches(lengths, seed * 1000 + epoch):
            step += 1
            wave = 0.5 * (1 + math.cos(math.pi * step / total))
            now = lr * min(1, step / warm) * wave
            for group in optimizer.param_groups:
                group["lr"] = now
            ids = X[rows][:, :int(lengths[rows].max())]    # 배치에서 가장 긴 문장까지만
            optimizer.zero_grad()
            if loss_of is None:
                loss = F.cross_entropy(net(ids), y[rows])
            else:
                loss = loss_of(net, ids)
            loss.backward()
            optimizer.step()
        if watch is not None:
            watch(epoch + 1, net)
    return net

count = lambda net: sum(p.numel() for p in net.parameters())
scores, nets, took = {}, {}, {}

def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def trial(name, make_net, data, lr, epochs=2, seeds=5, warm=200):
    """시드를 바꿔 학습시키고 정확도(%)를 scores[name]에 적는다."""
    X, y, X_test, y_test = data
    lap, scores[name], nets[name] = time.perf_counter(), [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        net = fit(make_net(), X, y, lr, seed, epochs, warm=warm)
        scores[name].append(accuracy(net, X_test, y_test))
        nets[name].append(net)
    scores[name], took[name] = np.array(scores[name]), time.perf_counter() - lap

def show(*names):
    print(pad("모델", 30) + "파라미터   정확도 (범위)           시간")
    for name in names:
        s = scores[name]
        span = f"({s.min():.2f}~{s.max():.2f})".ljust(15)
        print(pad(name, 30) + f"{count(nets[name][0]):>8,}   {np.median(s):>6.2f} "
              f"{span} {took[name]:>4.0f}초")

TASK = (T_tr, a_tr, T_te, a_te)
for kind, name in [("none", "위치 정보 없음"), ("learned", "학습되는 위치 임베딩"),
                   ("sinusoid", "사인 위치 인코딩")]:
    trial(name, lambda: TaskNet(kind), TASK, 0.003, epochs=5)
show("위치 정보 없음", "학습되는 위치 임베딩", "사인 위치 인코딩")
most = np.array([np.bincount(row, minlength=10).argmax() for row in numbers_te])
print(round((most == a_te.numpy()).mean() * 100, 2))     # 가장 많이 나온 숫자로 답할 때

In [ ]:
waves = sinusoid(64, 32)
print(waves.shape, waves.min().round(4), waves.max().round(4))
print(waves[0, :4].round(4), waves[1, :4].round(4), waves[2, :4].round(4))
print((1 / 10000 ** (np.arange(0, 32, 2) / 32))[[0, 1, 15]].round(6))
near = waves @ waves.T                                   # 두 위치의 벡터를 내적한 값
print(near[10, [10, 11, 12, 20, 40]].round(2))

## 12.3 트랜스포머 블록: 잔차 연결, 층 정규화, 완전 연결 층

In [ ]:
h = np.array([[[2.0, 4.0, 6.0, 8.0],                     # 문장 1, 토큰 1
               [1.0, 1.0, 1.0, 5.0]],                    # 문장 1, 토큰 2
              [[0.0, 0.0, 10.0, 10.0],                   # 문장 2, 토큰 1
               [3.0, 3.0, 3.0, 3.1]]])                   # 문장 2, 토큰 2
mean = h.mean(axis=-1, keepdims=True)                    # 토큰마다 숫자 4개의 평균
var = h.var(axis=-1, keepdims=True)
normed = (h - mean) / np.sqrt(var + 1e-5)
print(h.shape, mean[..., 0], var[..., 0])
print(normed.round(4))
by_torch = nn.LayerNorm(4)(torch.tensor(h, dtype=torch.float32)).detach().numpy()
print(float(np.abs(by_torch - normed).max()))

In [ ]:
class Block(nn.Module):
    """층 정규화 → 셀프 어텐션 → 더하기, 층 정규화 → 완전 연결 층 둘 → 더하기."""
    def __init__(self, dim, heads, scale=True, skip=True):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = SelfAttention(dim, heads, scale)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(),
                                 nn.Linear(4 * dim, dim))
        self.skip = skip

    def forward(self, x, hide=None, causal=False):
        mixed = self.attend(self.norm1(x), hide, causal)   # 다른 토큰의 정보를 가져온다
        x = x + mixed if self.skip else mixed
        changed = self.mlp(self.norm2(x))                    # 토큰마다 따로 계산한다
        return x + changed if self.skip else changed

class Encoder(nn.Module):
    """토큰 임베딩 + 위치 임베딩 → 블록 여러 개 → 층 정규화 → 선형층."""
    def __init__(self, vocab_size, outputs, dim=64, heads=4, layers=2, length=64,
                 position=True, **options):
        super().__init__()
        self.table = nn.Embedding(vocab_size, dim)
        self.where = nn.Embedding(length, dim) if position else None
        self.blocks = nn.ModuleList(Block(dim, heads, **options) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def features(self, ids, causal=False):
        x = self.table(ids)                                  # (배치, 길이, dim)
        if self.where is not None:
            x = x + self.where(torch.arange(ids.shape[1]))   # 위치 임베딩을 더한다
        hide = ids == PAD
        for block in self.blocks:
            x = block(x, hide, causal)
        return self.norm(x)

    def forward(self, ids):                                  # 문장마다 점수 outputs개
        x = self.features(ids)
        real = (ids != PAD)[:, :, None].float()              # 패딩을 빼고 평균한다
        return self.head((x * real).sum(dim=1) / real.sum(dim=1).clamp(min=1))

torch.manual_seed(0)
net = Encoder(V, outputs=2)
ids = X_tr[:8, :20]
x = net.table(ids) + net.where(torch.arange(20))
print(ids.shape, x.shape)
print(net.blocks[0](x).shape, net(ids).shape)
print(count(net.table), count(net.where), count(net.blocks[0].attend),
      count(net.blocks[0].mlp), count(net.blocks[0]), count(net))

In [ ]:
PART = (T_tr[:5000], a_tr[:5000], T_te, a_te)                # 문제 5,000개로 줄인다
deep = dict(dim=32, heads=1, layers=6, length=7)             # 블록을 6개 쌓는다
for skip in [True, False]:
    for warm in [200, 1]:                                    # warm=1이면 워밍업이 없다
        name = ("잔차 연결 있음" if skip else "잔차 연결 없음") + ", "
        name += "워밍업 있음" if warm > 1 else "워밍업 없음"
        trial(name, lambda: Encoder(17, 10, skip=skip, **deep), PART, 0.003, epochs=5,
              warm=warm)
show("잔차 연결 있음, 워밍업 있음", "잔차 연결 있음, 워밍업 없음",
     "잔차 연결 없음, 워밍업 있음", "잔차 연결 없음, 워밍업 없음")
print(scores["잔차 연결 없음, 워밍업 없음"].round(2))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

grams = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=2,
                        sublinear_tf=True)
logistic = LogisticRegression(max_iter=1000, C=3)
logistic.fit(grams.fit_transform(texts_tr), y_tr)
baseline = logistic.score(grams.transform(texts_va), y_va) * 100
print(f"글자 1~3-그램 로지스틱 회귀 {baseline:.2f}")

print("학습률     검증 정확도  (트랜스포머, 1에폭, 시드 0)")
for lr in [0.003, 0.01, 0.03]:
    torch.manual_seed(0)
    net = fit(Encoder(V, 2), X_tr, y_tr, lr, epochs=1)
    print(f"{lr:<8}   {accuracy(net, X_va, y_va):.2f}")

In [ ]:
REVIEWS = (X_tr, y_tr, X_va, y_va)
trial("트랜스포머", lambda: Encoder(V, 2), REVIEWS, 0.01)
trial("트랜스포머, 위치 정보 없음", lambda: Encoder(V, 2, position=False), REVIEWS,
      0.01)
show("트랜스포머", "트랜스포머, 위치 정보 없음")

def positive(net, texts):
    """긍정일 확률."""
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(encode(texts)), dim=1)[:, 1].numpy()

pair = ["지루하지 않고 재미있다", "재미있지 않고 지루하다"]
chances = {name: np.array([positive(net, pair) for net in nets[name]])
           for name in ["트랜스포머", "트랜스포머, 위치 정보 없음"]}
for name, both in chances.items():
    print(name)
    print(both.T.round(3))

rng = np.random.default_rng(0)
mixed_up = encode(["".join(rng.permutation(list(text[:64]))) for text in texts_va])
shuffled = {}
print("검증 정확도(%)               원래 문장   글자를 섞은 문장")
for name in chances:
    after = np.median([accuracy(net, mixed_up, y_va) for net in nets[name]])
    shuffled[name] = (np.median(scores[name]), after)
    print(pad(name, 28) + f"{shuffled[name][0]:.2f}       {after:.2f}")

## 12.4 다음 토큰을 예측하는 작은 언어 모델 만들기

In [ ]:
START, END = V, V + 1                                   # 문장의 시작과 끝을 알리는 토큰
names_of = {k: ch for ch, k in vocab.items()}
names_of.update({START: "<시작>", END: "<끝>", UNKNOWN: "<모름>"})

def lm_encode(texts, length=64):
    """<시작> + 글자 + <끝>. 길이는 length + 2칸이고 남는 칸은 0이다."""
    ids = torch.zeros(len(texts), length + 2, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [START] + [vocab.get(ch, UNKNOWN) for ch in text][:length] + [END]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

L_tr, L_va = lm_encode(texts_tr), lm_encode(texts_va)
one = lm_encode(["재밌다"])[0, :5]
print(L_tr.shape, L_va.shape, V + 2)
print("입력:", [names_of[k] for k in one[:-1].tolist()])
print("목표:", [names_of[k] for k in one[1:].tolist()])
print(int((L_tr[:, 1:] != PAD).sum()), int((L_va[:, 1:] != PAD).sum()))

In [ ]:
def pieces(texts):
    return [["<시작>"] + list(text)[:64] + ["<끝>"] for text in texts]

one_count = Counter()
two_count, three_count = defaultdict(Counter), defaultdict(Counter)
for line in pieces(texts_tr):
    for k in range(1, len(line)):
        before = line[k - 2] if k >= 2 else "<시작>"
        one_count[line[k]] += 1                              # 글자 하나의 횟수
        two_count[line[k - 1]][line[k]] += 1                 # 앞 글자 하나 뒤의 횟수
        three_count[before, line[k - 1]][line[k]] += 1       # 앞 글자 둘 뒤의 횟수
everything = sum(one_count.values())

def share(counter, token):
    return counter[token] / sum(counter.values()) if counter else 0.0

def ngram_loss(texts, mix):
    """세 확률을 mix의 비율로 섞은 확률의 음의 로그를 평균한다."""
    total, n = 0.0, 0
    for line in pieces(texts):
        for k in range(1, len(line)):
            before = line[k - 2] if k >= 2 else "<시작>"
            p1 = (one_count[line[k]] + 1) / (everything + V + 2)
            p2 = share(two_count.get(line[k - 1]), line[k])
            p3 = share(three_count.get((before, line[k - 1])), line[k])
            total -= math.log(mix[0] * p1 + mix[1] * p2 + mix[2] * p3)
            n += 1
    return total / n

floor = {"모든 토큰에 같은 확률": math.log(V + 2),
         "글자의 횟수 (1-그램)": ngram_loss(texts_va, (1, 0, 0)),
         "앞 글자 하나 (2-그램)": ngram_loss(texts_va, (0.1, 0.9, 0)),
         "앞 글자 둘 (3-그램)": ngram_loss(texts_va, (0.1, 0.3, 0.6))}
print("기준선                    검증 손실   퍼플렉서티")
for name, loss in floor.items():
    print(pad(name, 26) + f"{loss:.4f}   {math.exp(loss):>9.1f}")

In [ ]:
class LanguageModel(Encoder):
    """위치마다 다음 토큰의 점수 V + 2개를 낸다."""
    def __init__(self, causal=True, **options):
        super().__init__(V + 2, outputs=V + 2, length=66, **options)
        self.causal = causal

    def forward(self, ids):                                  # (배치, 길이, V + 2)
        return self.head(self.features(ids, causal=self.causal))

def lm_loss(net, ids):
    """앞 토큰들로 다음 토큰을 맞히는 교차 엔트로피. 패딩인 목표는 뺀다."""
    out = net(ids[:, :-1])                                   # 입력은 마지막 칸을 뺀 것
    goal = ids[:, 1:]                                        # 목표는 한 칸 뒤의 토큰
    return F.cross_entropy(out.reshape(-1, out.shape[-1]), goal.reshape(-1),
                           ignore_index=PAD)

def lm_score(net, data):
    """토큰 하나의 평균 손실."""
    net.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for k in range(0, len(data), 500):
            ids = data[k:k + 500]
            out = net(ids[:, :-1])
            total += F.cross_entropy(out.reshape(-1, out.shape[-1]),
                                     ids[:, 1:].reshape(-1), ignore_index=PAD,
                                     reduction="sum").item()
            n += (ids[:, 1:] != PAD).sum().item()
    return total / n

torch.manual_seed(0)
fresh, peeking = LanguageModel(), LanguageModel(causal=False)
peeking.load_state_dict(fresh.state_dict())               # 같은 가중치, 마스크만 다르다
ids = L_va[:1, :12].clone()
changed = ids.clone()
changed[0, 11] = vocab["다"]                                 # 마지막 토큰 하나를 바꾼다
fresh.eval(), peeking.eval()
with torch.no_grad():
    for name, net in [("인과 마스크 있음", fresh), ("인과 마스크 없음", peeking)]:
        gap = (net(ids) - net(changed)).abs().amax(dim=-1)[0]    # 위치마다 출력의 차이
        print(name, gap[:11].max().item(), round(gap[11].item(), 4))
print(count(fresh), round(lm_score(fresh, L_va[:2000]), 4), round(math.log(V + 2), 4))

In [ ]:
print("학습률     검증 손실  (언어 모델, 1에폭, 시드 0)")
for lr in [0.01, 0.03, 0.1]:
    torch.manual_seed(0)
    net = fit(LanguageModel(), L_tr, None, lr, epochs=1, loss_of=lm_loss)
    print(f"{lr:<8}   {lm_score(net, L_va):.4f}")

losses, models = {}, {}

def lm_trial(name, make_net, lr=0.03, epochs=2, seeds=1):
    lap, losses[name], models[name] = time.perf_counter(), [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        net = fit(make_net(), L_tr, None, lr, seed, epochs, loss_of=lm_loss)
        losses[name].append((lm_score(net, L_tr[:5000]), lm_score(net, L_va)))
        models[name].append(net)
    losses[name], took[name] = np.array(losses[name]), time.perf_counter() - lap

def lm_show(*names):
    print(pad("모델", 24) + "학습 손실  검증 손실 (범위)        퍼플렉서티   시간")
    for name in names:
        tr, va = losses[name][:, 0], losses[name][:, 1]
        span = f"({va.min():.4f}~{va.max():.4f})".ljust(18) if len(va) > 1 else " " * 18
        print(pad(name, 24) + f"{np.median(tr):.4f}     {np.median(va):.4f} {span}"
              f"{math.exp(np.median(va)):>7.1f}  {took[name]:>4.0f}초")

lm_trial("언어 모델", lambda: LanguageModel(), seeds=5)
lm_show("언어 모델")

In [ ]:
lm_trial("인과 마스크 없음", lambda: LanguageModel(causal=False), epochs=1)
lm_trial("제곱근으로 나누지 않음", lambda: LanguageModel(scale=False))
lm_trial("위치 정보 없음", lambda: LanguageModel(position=False))
lm_show("언어 모델", "인과 마스크 없음", "제곱근으로 나누지 않음", "위치 정보 없음")
print(losses["언어 모델"][0].round(4))

## 12.5 문장 생성: 온도와 표본 추출

In [ ]:
logits = np.array([2.0, 1.0, 0.0])                       # 다음 토큰 후보 셋의 점수
for T in [0.5, 1.0, 2.0]:
    print(T, softmax(logits / T).round(4))
print(softmax(logits / 0.01).round(4), softmax(logits / 100).round(4))

In [ ]:
def generate(net, start, T, seed, longest=30):
    """start 뒤에 토큰을 하나씩 뽑아 붙인다. T가 0이면 확률이 가장 큰 토큰을 고른다."""
    picker = torch.Generator().manual_seed(seed)
    ids = [START] + [vocab[ch] for ch in start]
    net.eval()
    for _ in range(longest):
        with torch.no_grad():
            out = net(torch.tensor([ids]))[0, -1]            # 마지막 위치의 점수만 쓴다
        out[[PAD, UNKNOWN, START]] = float("-inf")           # 뽑지 않을 토큰
        if T == 0:
            token = int(out.argmax())
        else:
            token = int(torch.multinomial(torch.softmax(out / T, dim=0), 1,
                                          generator=picker))
        if token == END:
            break
        ids.append(token)
    return "".join(names_of[k] for k in ids[1:])

writer = models["언어 모델"][0]
for start in ["이 영화", "내용이"]:
    for T in [0, 0.5, 1.0, 1.5]:
        for seed in range(2):
            print(f"T={T:<3} 시드 {seed} | " + generate(writer, start, T, seed))
print(generate(models["인과 마스크 없음"][0], "이 영화", 0.5, 0) == "이 영화")

In [ ]:
few = L_tr[:200]                                            # 학습 문장을 200개로 줄인다
history = []
torch.manual_seed(0)
lap = time.perf_counter()
parrot = fit(LanguageModel(), few, None, 0.003, epochs=120, loss_of=lm_loss, warm=50,
             watch=lambda epoch, net: epoch % 10 == 0 and history.append(
                 (epoch, lm_score(net, few), lm_score(net, L_va[:2000]))))
print(f"{time.perf_counter() - lap:.0f}초")
for epoch, on_train, on_valid in history[::3]:
    print(f"{epoch:>4}에폭  학습 손실 {on_train:.4f}  검증 손실 {on_valid:.4f}")

seen = set(texts_tr[:200])
starts = sorted({text[:2] for text in texts_tr[:200] if len(text) > 12})[:40]
copied = {}
for name, net in [("문장 200개로 학습한 모델", parrot),
                  ("문장 30,000개로 학습한 모델", writer)]:
    made = [generate(net, start, 0, 0, longest=64) for start in starts]
    copied[name] = sum(text in seen for text in made)
    print(pad(name, 30) + f"{copied[name]}/{len(starts)}")

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 12-3
from matplotlib.patches import Rectangle
fig = plt.figure(figsize=(7, 9.2))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 2.1], hspace=0.36, wspace=0.12)
ax = fig.add_subplot(grid[0, :])
kinds = [("위치 정보 없음", GRAY), ("학습되는 위치 임베딩", NAVY), ("사인 위치 인코딩", TEAL)]
rule = (most == a_te.numpy()).mean() * 100
for row, (label, shade) in enumerate(kinds):
    marks = scores[label]
    ax.barh(row, np.median(marks), color=shade, height=0.58)
    ax.scatter(marks, [row] * len(marks), s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(max(marks) + 1.8, row, f"{np.median(marks):.2f}%", va="center", fontsize=10.5, weight="bold")
ax.axvline(rule, color=BRICK, lw=0.9, ls="--"); ax.axvline(10, color="#666", lw=0.9, ls="--")
ax.text(rule + 1.2, 0.5, f"가장 많이 나온 숫자를 답하는 규칙 {rule:.2f}%", fontsize=9, color=BRICK, va="center")
ax.text(10.8, 0.5, "아무 숫자나 10%", fontsize=9, color="#555", va="center")
ax.set_yticks(range(3)); ax.set_yticklabels([label for label, _ in kinds], fontsize=10)
ax.invert_yaxis(); ax.set_xlim(0, 116); ax.set_xlabel("시험 문제 5,000개의 정확도 (%)", fontsize=9)
frame(ax, grid="x")
ax.set_title("① 질문한 칸의 숫자를 답하는 과제 (막대는 시드 5개의 중앙값, 흰 점은 시드 하나의 값)", fontsize=10, loc="left",
             color=NAVY, weight="bold", x=-0.24)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
shown = np.arange(8)
looks = {}
for col, label in enumerate(["위치 정보 없음", "학습되는 위치 임베딩"]):
    ax = fig.add_subplot(grid[1, col])
    probe = nets[label][0]
    with torch.no_grad():
        probe(T_te[shown])
    looks[label] = probe.attend.maps[:, 0, 0, 1:].numpy()            # 질문 칸이 항목 여섯 칸에 준 가중치
    ax.imshow(looks[label], cmap="Blues", vmin=0, vmax=1.25, aspect="auto")
    for row in range(8):
        for item in range(6):
            share = looks[label][row, item]
            asked = item == where_te[shown[row]].item()
            ink = "white" if share > 0.6 else "#222"
            ax.text(item, row - 0.17, f"{T_te[shown[row], item + 1].item() - 1}", ha="center", va="center", fontsize=10.5,
                    color=ink, weight="bold")
            ax.text(item, row + 0.24, f"{share:.2f}", ha="center", va="center", fontsize=9, color=ink)
            if asked:
                ax.add_patch(Rectangle((item - 0.47, row - 0.46), 0.94, 0.92, fill=False, ec=BRICK, lw=2.0))
        if col == 0:
            ax.text(-0.75, row, f"{where_te[shown[row]].item() + 1}번 칸", ha="right", va="center", fontsize=9, clip_on=False)
    ax.set_xticks(range(6)); ax.set_xticklabels([f"{k}번" for k in range(1, 7)], fontsize=9)
    ax.xaxis.tick_top(); ax.set_yticks([]); ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    hit = (probe.attend.maps[:, 0, 0, 1:].argmax(dim=1) == where_te[shown]).sum().item()
    ax.set_title(("② 위치 정보 없음" if col == 0 else "③ 위치 임베딩 있음") + f": 맞게 본 문제 {hit}/8", fontsize=9.6, loc="left",
                 color=NAVY, weight="bold", pad=24)
    if col == 0:
        ax.text(-0.75, -0.9, "질문", ha="right", fontsize=9, color=NAVY, weight="bold", clip_on=False)
fig.text(0.5, 0.008, "②와 ③: 큰 숫자는 그 칸에 놓인 숫자, 작은 숫자는 질문 칸이 그 칸에 준 어텐션 가중치. 벽돌색 테두리가 질문한 칸이다.\n제목의 ‘맞게 본 문제’는 가중치가 가장 큰 칸이 질문한 칸과 같은 문제의 수다",
         ha="center", fontsize=9, color="#444", linespacing=1.5)
fig.subplots_adjust(left=0.2, right=0.97, top=0.955, bottom=0.06)
fig.savefig("fig_12_3.png", dpi=300)
plt.show()
print([[round(float(v), 2) for v in row] for row in looks["위치 정보 없음"][[3, 5, 6]]],
      [round(float(v), 2) for v in looks["학습되는 위치 임베딩"].max(axis=1)])
print([int((looks[label].argmax(axis=1) == where_te[:8].numpy()).sum()) for label in looks], where_te[:8].tolist(), round(float(rule), 2))

In [ ]:
# 그림 12-4
fig, axes = plt.subplots(3, 1, figsize=(7, 9.2), gridspec_kw={"height_ratios": [1.25, 1, 0.9]})
ax = axes[0]
picture = ax.imshow(waves.T, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto", origin="lower")
ax.set_xlabel("위치 (문장의 몇 번째 칸인가)", fontsize=9); ax.set_ylabel("벡터의 몇 번째 숫자인가", fontsize=9)
ax.set_xticks(range(0, 64, 8)); ax.set_yticks([0, 8, 16, 24, 31])
bar = fig.colorbar(picture, ax=ax, pad=0.05, ticks=[-1, 0, 1]); bar.ax.set_yticklabels(["$-$1", "0", "1"], fontsize=9)
for line_at, shade in [(0, NAVY), (8, TEAL), (16, BRICK)]:
    ax.annotate("", xy=(63.6, line_at), xytext=(66.6, line_at), arrowprops=dict(arrowstyle="-|>", color=shade, lw=1.6),
                annotation_clip=False)
ax.set_title("① 위치 인코딩 표: 세로 한 줄이 한 위치의 벡터(숫자 32개)다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
for column, shade, dash in [(0, NAVY, "-"), (8, TEAL, "--"), (16, BRICK, ":")]:
    ax.plot(range(64), waves[:, column], color=shade, ls=dash, lw=1.8, marker="o", ms=2.6,
            label=f"{column}번째 숫자 (한 바퀴에 {2 * np.pi * 10000 ** (column / 32):,.1f}칸)")
ax.set_xlim(-1, 64); ax.set_ylim(-1.25, 1.95); ax.set_xticks(range(0, 64, 8))
ax.set_yticks([-1, 0, 1]); ax.set_yticklabels(["$-$1", "0", "1"])
ax.set_xlabel("위치", fontsize=9); ax.set_ylabel("값", fontsize=9)
ax.legend(frameon=False, fontsize=8.8, loc="upper center", ncol=3, columnspacing=0.9, handlelength=1.8)
frame(ax)
ax.set_title("② ①의 화살표 세 줄을 꺼낸 것: 아래쪽 숫자일수록 빠르게, 위쪽일수록 느리게 변한다", fontsize=10, loc="left",
             color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[2]
closeness = waves @ waves[10]
ax.bar(range(64), closeness, color=[BRICK if k == 10 else NAVY for k in range(64)], width=0.8)
ax.text(8.8, closeness[10] + 2.2, f"위치 10 자신: {closeness[10]:.0f}", ha="right", fontsize=9, color=BRICK, weight="bold")
for other in [11, 20, 40]:
    ax.annotate(f"위치 {other}: {closeness[other]:.2f}", xy=(other, closeness[other]), xytext=(other + 6, closeness[other] + 4.4),
                fontsize=9, color="#222", arrowprops=dict(arrowstyle="->", color="#555", lw=0.8))
ax.set_xlim(-1, 64); ax.set_ylim(0, 22); ax.set_xticks(range(0, 64, 8))
ax.set_xlabel("위치", fontsize=9); ax.set_ylabel("내적", fontsize=9)
frame(ax)
ax.set_title("③ 위치 10의 벡터와 다른 위치의 벡터를 내적한 값: 가까운 위치일수록 크다", fontsize=10, loc="left", color=NAVY,
             weight="bold")
ax.tick_params(labelsize=9)
fig.subplots_adjust(left=0.1, right=0.99, top=0.965, bottom=0.055, hspace=0.42)
fig.savefig("fig_12_4.png", dpi=300)
plt.show()
print(closeness[[10, 11, 20, 40]].round(2), [round(2 * np.pi * 10000 ** (c / 32), 1) for c in (0, 8, 16)])

In [ ]:
# 그림 12-6
from matplotlib.patches import Patch
fig, axes = plt.subplots(3, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [0.62, 1.15, 1.15]})
ax = axes[0]
all_steps = 2 * math.ceil(30000 / 64)
walk = np.arange(1, all_steps + 1)
plan = 0.01 * np.minimum(1, walk / 200) * 0.5 * (1 + np.cos(np.pi * walk / all_steps))
ax.plot(walk, plan, color=BRICK, lw=2)
ax.axvline(200, color="#888", lw=0.9, ls=":")
top_at = int(plan.argmax())
ax.annotate(f"{walk[top_at]}걸음에서 가장 높다 ({plan[top_at]:.4f})", xy=(walk[top_at], plan[top_at]), xytext=(330, 0.0098),
            fontsize=9, color=BRICK, va="center", arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.text(100, 0.0012, "워밍업\n(200걸음)", ha="center", fontsize=9, color="#444", linespacing=1.3)
ax.text(600, 0.0072, "코사인 모양으로 0까지 줄인다", ha="center", fontsize=9, color="#444")
ax.set_xlim(0, all_steps); ax.set_ylim(0, 0.0115); ax.set_yticks([0, 0.005, 0.01])
ax.set_xlabel(f"걸음 (2에폭 = {all_steps}걸음)", fontsize=9); ax.set_ylabel("학습률", fontsize=9)
frame(ax)
ax.set_title("① 학습률 일정: 처음 200걸음 동안 올리고 그 뒤로 줄인다", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
duo = ["트랜스포머", "트랜스포머, 위치 정보 없음"]
for spot, label in enumerate(duo):
    kept, mixed = shuffled[label]
    ax.bar(spot - 0.19, kept, width=0.36, color=NAVY); ax.bar(spot + 0.19, mixed, width=0.36, color=BRICK)
    ax.scatter([spot - 0.19] * 5, scores[label], s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(spot - 0.19, max(scores[label]) + 0.45, f"{kept:.2f}", ha="center", fontsize=10, color=NAVY, weight="bold")
    ax.text(spot + 0.19, mixed + 0.55, f"{mixed:.2f}", ha="center", fontsize=10, color=BRICK, weight="bold")
    gap = mixed - kept
    ax.text(spot, 72.6, "변화 없음" if abs(gap) < 0.005 else f"섞으면 {gap:+.2f}%p".replace("-", "$-$"), ha="center", fontsize=9.6,
            color="#111", weight="bold", bbox=BOX)
ax.axhline(baseline, color="#333", lw=1.0, ls="--")
ax.text(1.48, baseline - 1.0, f"글자 1~3-그램 로지스틱 회귀 {baseline:.2f}%", ha="right", fontsize=9, color="#333")
ax.set_xticks(range(2)); ax.set_xticklabels(["트랜스포머 (위치 임베딩 있음)", "트랜스포머, 위치 정보 없음"], fontsize=10)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(72, 89.5); ax.set_ylabel("검증 정확도 (%)", fontsize=9)
ax.legend(handles=[Patch(color=NAVY, label="원래 문장 (흰 점은 시드 하나)"), Patch(color=BRICK, label="글자를 섞은 문장")], frameon=False,
          fontsize=9, loc="upper left", ncol=2)
frame(ax)
ax.set_title("② 영화평 분류의 검증 정확도 (시드 5개의 중앙값. 세로축은 72%에서 시작)", fontsize=10, loc="left", color=NAVY,
             weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[2]
for block_no, label in enumerate(duo):
    for seed in range(5):
        spot = block_no * 6.2 + seed
        good, bad = chances[label][seed]
        ax.plot([spot, spot], [good, bad], color="#9aa1ab", lw=1.4, zorder=2)
        ax.scatter(spot, good, s=64, color=TEAL, zorder=4, marker="o")
        ax.scatter(spot, bad, s=64, color=BRICK, zorder=5, marker="s" if abs(good - bad) > 1e-6 else "x")
        right = good > 0.5 and bad < 0.5
        ax.text(spot, -0.1, "맞음" if right else "틀림", ha="center", fontsize=9, color=TEAL if right else "#8a929c",
                weight="bold" if right else "normal")
        ax.text(spot, -0.2, f"시드 {seed}", ha="center", fontsize=7.6, color="#555")
    ax.text(block_no * 6.2 + 2, 1.07, ["위치 임베딩 있음", "위치 정보 없음 (두 점이 겹친다)"][block_no], ha="center", fontsize=9.6,
            color=NAVY, weight="bold")
ax.axhline(0.5, color="#666", lw=0.9, ls="--"); ax.axvline(5.1, color="#c9ced6", lw=0.8)
ax.text(10.75, 0.52, "0.5", fontsize=9, color="#555", ha="right")
ax.scatter([], [], s=64, color=TEAL, marker="o", label=f"“{pair[0]}” (0.5보다 커야 맞다)")
ax.scatter([], [], s=64, color=BRICK, marker="s", label=f"“{pair[1]}” (0.5보다 작아야 맞다)")
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0.0, 0.93), handletextpad=0.3)
ax.set_xticks([]); ax.set_xlim(-0.8, 10.9); ax.set_ylim(-0.26, 1.16); ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.set_ylabel("긍정일 확률", fontsize=9)
frame(ax)
ax.set_title("③ 같은 글자로 만든, 뜻이 반대인 두 문장에 시드마다 준 확률", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.03, hspace=0.36)
fig.savefig("fig_12_6.png", dpi=300)
plt.show()
print(int(walk[top_at]), round(float(plan[top_at]), 4), all_steps,
      [int(g > 0.5 and b < 0.5) for g, b in chances["트랜스포머"]])

In [ ]:
# 그림 12-8
fig, ax = plt.subplots(figsize=(7, 7.0))
lines = [(name, [loss], GRAY, "횟수를 센 모델") for name, loss in floor.items()]
lines += [("언어 모델 (트랜스포머)", losses["언어 모델"][:, 1], NAVY, "시드 5개"),
          ("제곱근으로 나누지 않음", losses["제곱근으로 나누지 않음"][:, 1], NAVY, "시드 0"),
          ("위치 정보 없음", losses["위치 정보 없음"][:, 1], NAVY, "시드 0"),
          ("인과 마스크 없음", losses["인과 마스크 없음"][:, 1], BRICK, "시드 0")]
heights = [0, 1, 2, 3, 4.7, 5.7, 6.7, 8.4]
for row, (label, marks, shade, note_) in zip(heights, lines):
    middle = float(np.median(marks))
    ax.barh(row, middle, color=shade, height=0.64)
    if len(marks) > 1:
        ax.scatter(marks, [row] * len(marks), s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(max(marks) + 0.09, row, f"{middle:.4f}", va="center", fontsize=10.5, color="#111", weight="bold")
    ax.text(max(marks) + 1.3, row, f"퍼플렉서티 {math.exp(middle):,.1f}", va="center", fontsize=9, color="#444")
    ax.text(-0.08, row - 0.12, label, va="center", ha="right", fontsize=9.6, color="#111", weight="bold")
    ax.text(-0.08, row + 0.25, note_, va="center", ha="right", fontsize=9, color="#5b6470")
best_count = floor["앞 글자 둘 (3-그램)"]
ax.axvline(best_count, color="#333", lw=1.0, ls="--", ymin=0.2)
ax.text(best_count + 0.06, 3.86, f"3-그램 {best_count:.4f}", fontsize=9, color="#333", va="center")
ax.text(0.5, 7.55, "뒤의 토큰을 볼 수 있어서 나온 값이다. 다음 토큰을 예측한 것이 아니다", fontsize=9, color=BRICK, va="center")
ax.text(3.25, -0.72, "신경망을 쓰지 않는 기준선", fontsize=9, color="#5b6470", weight="bold")
ax.text(0.02, 3.98, "트랜스포머 (2에폭)", fontsize=9, color=NAVY, weight="bold")
ax.set_yticks([]); ax.set_ylim(9.0, -1.05); ax.set_xlim(0, 10.8)
ax.set_xlabel("검증 문장의 토큰 하나당 손실 (작을수록 좋다)", fontsize=9)
frame(ax, grid="x")
ax.set_title("다음 글자를 예측하는 모델의 검증 손실", fontsize=10, loc="left", color=NAVY, weight="bold", x=-0.38)
ax.tick_params(axis="x", labelsize=9)
fig.subplots_adjust(left=0.29, right=0.97, top=0.945, bottom=0.08)
fig.savefig("fig_12_8.png", dpi=300)
plt.show()
print([(label, round(float(np.median(marks)), 4), round(math.exp(float(np.median(marks))), 1)) for label, marks, _, _ in lines])

In [ ]:
# 그림 12-9
from matplotlib.patches import Rectangle
fig = plt.figure(figsize=(7, 9.8))
layout = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.62], hspace=0.3, wspace=0.16)
spoken = "이 영화 정말 재밌다"
tokens = [START] + [vocab[ch] for ch in spoken]
writer.eval()
with torch.no_grad():
    writer(torch.tensor([tokens]))
head_maps = writer.blocks[0].attend.maps[0].numpy()                  # (헤드 4개, 12, 12)
tags = ["시작"] + ["빈칸" if ch == " " else ch for ch in spoken]
size = len(tags)
far = {}                                                            # 헤드마다, 몇 칸 앞의 토큰에 준 가중치의 평균
for head in range(4):
    by_gap = [np.mean([head_maps[head, t, t - gap] for t in range(gap, size)]) for gap in range(4)]
    by_gap.append(np.mean([head_maps[head, t, :t - 3].sum() for t in range(4, size)]))
    far[head] = np.array(by_gap)
gap_names = ["자기 자신", "1칸 앞", "2칸 앞", "3칸 앞", "4칸 이상 앞"]
for head in range(4):
    ax = fig.add_subplot(layout[head // 2, head % 2])
    shown_map = np.ma.masked_where(np.triu(np.ones_like(head_maps[head]), 1) > 0, head_maps[head])
    picture = ax.imshow(shown_map, cmap="Blues", vmin=0, vmax=1)
    for row in range(size):
        peak = int(head_maps[head, row].argmax())
        for col in range(row + 1):
            share = head_maps[head, row, col]
            if share >= 0.095 or col == peak:
                ax.text(col, row, "1.0" if share > 0.995 else f"{share:.2f}"[1:], ha="center", va="center",
                        fontsize=6.4 if col == peak else 5.6, weight="bold" if col == peak else "normal",
                        color="white" if share > 0.55 else "#1b2733")
        ax.add_patch(Rectangle((peak - 0.5, row - 0.5), 1, 1, fill=False, ec=BRICK, lw=1.1))
    ax.set_xticks(range(size)); ax.set_yticks(range(size))
    ax.set_xticklabels(tags, fontsize=8); ax.set_yticklabels(tags, fontsize=8)
    ax.xaxis.tick_top(); ax.tick_params(length=0, pad=1.5)
    for label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        if label.get_text() in ("시작", "빈칸"):
            label.set_fontsize(6); label.set_color("#8a929c")
    ax.set_xticks(np.arange(-0.5, size), minor=True); ax.set_yticks(np.arange(-0.5, size), minor=True)
    ax.grid(which="minor", color="white", lw=0.6); ax.tick_params(which="minor", length=0)
    ax.set_facecolor("#e9ebee")
    for side in ax.spines.values():
        side.set_visible(False)
    most_gap = int(far[head].argmax())
    ax.set_title(f"헤드 {head + 1}", fontsize=10.5, loc="left", color=NAVY, weight="bold", pad=17)
    ax.text(1.0, 1.085, f"가장 많이 보는 곳: {gap_names[most_gap]} (평균 {far[head][most_gap]:.2f})", transform=ax.transAxes,
            ha="right", fontsize=8, color=BRICK, weight="bold")
    ax.text(8.6, 2.3, "회색: 뒤의 토큰이라\n인과 마스크로 가려진 칸\n(가중치 0)", fontsize=7.2, color="#5b6470", ha="center", va="center",
            linespacing=1.45)
    if head % 2 == 0:
        ax.set_ylabel("질의 (지금 계산하는 토큰)", fontsize=8.4)
    if head == 3:
        ax.annotate("‘화’가 바로 앞의\n‘영’에 준 가중치", xy=(3.55, 4.0), xytext=(8.4, 5.9), fontsize=7.2, color="#222", ha="center",
                    va="center", linespacing=1.4, arrowprops=dict(arrowstyle="->", color="#444", lw=0.8))
ax = fig.add_subplot(layout[2, :])
shades = [NAVY, TEAL, GRAY, BRICK]
for head in range(4):
    spots = np.arange(5) + (head - 1.5) * 0.2
    ax.bar(spots, far[head], width=0.19, color=shades[head], label=f"헤드 {head + 1}")
    for spot, share in zip(spots, far[head]):
        if share >= 0.3:
            ax.text(spot, share + 0.02, f"{share:.2f}"[1:], ha="center", fontsize=8.4, color=shades[head], weight="bold")
ax.set_xticks(range(5)); ax.set_xticklabels(gap_names[:4] + ["4칸 이상 앞\n(여러 칸의 합)"], fontsize=9.6)
ax.set_ylim(0, 1.3); ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0]); ax.set_ylabel("가중치의 평균", fontsize=9)
ax.legend(frameon=False, fontsize=9, ncol=4, loc="upper center", columnspacing=1.4, handlelength=1.2)
frame(ax)
ax.set_title("질의에서 몇 칸 앞의 토큰을 보는가: 위의 네 그림을 헤드마다 한 줄로 줄인 것", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=8.6); ax.tick_params(axis="x", length=0)
fig.text(0.5, 0.008, "위의 네 그림: 행이 질의, 열이 키(가져올 토큰). 칸이 진할수록 가중치가 크다. 0.10 이상인 값을 적었고, 행마다 가장 큰 칸에 벽돌색 테두리를 쳤다",
         ha="center", fontsize=8.2, color="#444")
fig.subplots_adjust(left=0.085, right=0.985, top=0.94, bottom=0.075)
fig.savefig("fig_12_9.png", dpi=300)
plt.show()
print(head_maps.shape, float(np.triu(head_maps, 1).max()), head_maps.sum(axis=-1).round(3).min(),
      [int(head_maps[h, -1].argmax()) for h in range(4)], head_maps[3, 3, 2].round(2), head_maps[3, 4, 3].round(2))
print({h + 1: [round(float(v), 2) for v in far[h]] for h in range(4)})

In [ ]:
# 그림 12-10
fig, axes = plt.subplots(2, 1, figsize=(7, 8.6))
ax = axes[0]
heats = [(0.5, NAVY), (1.0, TEAL), (2.0, BRICK)]
for shift, (T, shade) in zip([-0.27, 0, 0.27], heats):
    shares = softmax(logits / T)
    ax.bar(np.arange(3) + shift, shares, width=0.25, color=shade, label=f"T = {T}")
    for spot, share in enumerate(shares):
        ax.text(spot + shift, share + 0.02, f"{share:.3f}", ha="center", fontsize=9, color=shade, weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(["후보 1 (점수 2)", "후보 2 (점수 1)", "후보 3 (점수 0)"], fontsize=10)
ax.set_ylim(0, 1.05); ax.set_ylabel("뽑힐 확률", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="upper right", title="온도", title_fontsize=9)
ax.text(1.0, 0.82, "온도가 낮으면 점수가 큰 후보에 몰리고,\n온도가 높으면 세 후보가 비슷해진다", ha="center", fontsize=9, color="#222", linespacing=1.5)
frame(ax)
ax.set_title("① 점수가 2, 1, 0인 세 후보를 온도 세 가지로 확률로 바꾼 결과", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
asked_ids = torch.tensor([[START] + [vocab[ch] for ch in "이 영화"]])
with torch.no_grad():
    after = writer(asked_ids)[0, -1].clone()
after[[PAD, UNKNOWN, START]] = float("-inf")
plain = torch.softmax(after, dim=0)
leaders = torch.topk(plain, 8).indices.tolist()
for shift, (T, shade) in zip([-0.27, 0, 0.27], [(0.5, NAVY), (1.0, TEAL), (1.5, BRICK)]):
    shares = torch.softmax(after / T, dim=0)[leaders].numpy()
    ax.bar(np.arange(8) + shift, shares, width=0.25, color=shade, label=f"T = {T}")
    ax.text(shift + (0.0 if shift < 0 else 0.07), shares[0] + 0.015, f"{shares[0]:.2f}", ha="center" if shift < 0 else "left", fontsize=9, color=shade, weight="bold")
ax.set_xticks(range(8))
ax.set_xticklabels([{" ": "빈칸", "<끝>": "끝"}.get(names_of[k], names_of[k]) for k in leaders], fontsize=11, weight="bold")
ax.set_ylabel("뽑힐 확률", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="upper right", title="온도", title_fontsize=9)
rest = [1 - torch.softmax(after / T, dim=0)[leaders].sum().item() for T in (0.5, 1.0, 1.5)]
ax.text(7.45, ax.get_ylim()[1] * 0.42, "여기 없는 나머지 토큰의 확률을 모두 더하면\n" +
        f"T = 0.5일 때 {rest[0]:.3f}, 1.0일 때 {rest[1]:.3f}, 1.5일 때 {rest[2]:.3f}", ha="right", fontsize=9, color="#222", linespacing=1.5)
frame(ax)
ax.set_title("② 언어 모델이 “이 영화” 다음 글자로 낸 확률 (T = 1에서 가장 큰 8개)", fontsize=10, loc="left", color=NAVY,
             weight="bold")
ax.tick_params(axis="y", labelsize=9); ax.tick_params(axis="x", length=0)
fig.subplots_adjust(left=0.1, right=0.97, top=0.96, bottom=0.05, hspace=0.26)
fig.savefig("fig_12_10.png", dpi=300)
plt.show()
print([names_of[k] for k in leaders], plain[leaders].numpy().round(3), np.round(rest, 3))
print([round(torch.softmax(after / T, dim=0)[leaders[0]].item(), 2) for T in (0.5, 1.0, 1.5)])

In [ ]:
# 그림 12-11
fig, axes = plt.subplots(2, 1, figsize=(7, 8.6), gridspec_kw={"height_ratios": [1.7, 1]})
ax = axes[0]
epochs_seen = [epoch for epoch, _, _ in history]
ax.plot(epochs_seen, [on_train for _, on_train, _ in history], color=NAVY, lw=2.2, marker="o", ms=4, label="학습 문장 200개의 손실")
ax.plot(epochs_seen, [on_valid for _, _, on_valid in history], color=BRICK, lw=2.2, marker="s", ms=4, ls="--",
        label="검증 문장 2,000개의 손실")
flat = floor["모든 토큰에 같은 확률"]
ax.axhline(flat, color="#666", lw=0.9, ls=":")
ax.text(2, flat + 0.18, f"모든 토큰에 같은 확률을 줄 때 {flat:.2f}", ha="left", fontsize=9, color="#444")
good = float(np.median(losses["언어 모델"][:, 1]))
ax.axhline(good, color=TEAL, lw=1.0, ls="--")
ax.text(epochs_seen[-1], good + 0.16, f"문장 30,000개로 학습한 모델의 검증 손실 {good:.2f}", ha="right", fontsize=9, color=TEAL)
last_epoch, last_train, last_valid = history[-1]
ax.annotate(f"{last_train:.2f}", xy=(last_epoch, last_train), xytext=(last_epoch - 9, last_train + 0.75), fontsize=10, color=NAVY,
            weight="bold", arrowprops=dict(arrowstyle="->", color=NAVY, lw=0.9))
ax.annotate(f"{last_valid:.2f}", xy=(last_epoch, last_valid), xytext=(last_epoch - 9, last_valid + 0.6), fontsize=10, color=BRICK,
            weight="bold", arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.set_xlim(0, epochs_seen[-1] + 3); ax.set_ylim(0, 9.4)
ax.set_xlabel("에폭", fontsize=9); ax.set_ylabel("토큰 하나당 손실", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="center right", bbox_to_anchor=(1.0, 0.56))
frame(ax)
ax.set_title("① 문장 200개만으로 학습시킨 언어 모델: 학습 손실은 내려가고 검증 손실은 올라간다", fontsize=10, loc="left",
             color=NAVY, weight="bold")
ax.tick_params(labelsize=9)
ax = axes[1]
for row, (label, shade) in enumerate([("문장 200개로 학습한 모델", BRICK), ("문장 30,000개로 학습한 모델", NAVY)]):
    ax.barh(row, copied[label], color=shade, height=0.56)
    ax.text(copied[label] + 0.6, row, f"{copied[label]}개 / {len(starts)}개", va="center", fontsize=10.5, weight="bold", color=shade)
ax.set_yticks(range(2)); ax.set_yticklabels(["문장 200개로\n학습한 모델", "문장 30,000개로\n학습한 모델"], fontsize=9.6)
ax.invert_yaxis(); ax.set_xlim(0, len(starts) * 1.25)
ax.set_xlabel(f"학습 문장의 앞 두 글자를 주고 만든 문장 {len(starts)}개 가운데, 학습 문장과 글자 하나까지 같은 것의 수", fontsize=9)
frame(ax, grid="x")
ax.set_title("② 만든 문장이 학습 문장을 그대로 옮긴 것인가", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.17, right=0.97, top=0.96, bottom=0.07, hspace=0.28)
fig.savefig("fig_12_11.png", dpi=300)
plt.show()
print(history[-1][0], round(last_train, 4), round(last_valid, 4), copied, len(starts))
print(min(history, key=lambda row: row[2])[0], round(min(row[2] for row in history), 2))